# Chapter 5 — Normal distributions & the Central Limit Theorem

Companion notebook to `docs/models/ch05_normal.md` and the Streamlit page `app/pages/05_Normal_Distribution.py`.

**Goal.** Show that the distribution of *sample means* converges to a normal, no matter what the underlying source looks like — and demonstrate the √n scaling rule.


In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from modelthinker.distributions.normal import simulate, sqrt_n_scan

## 1. The CLT in one picture

Draw many sample means and overlay the normal the CLT predicts.

In [ ]:
r = simulate(source='exponential', n=30, num_samples=10_000, seed=42)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(r.sample_means, bins=60, density=True, alpha=0.7, label=f'sample means (n={r.n})')
xs = np.linspace(r.sample_means.min(), r.sample_means.max(), 400)
ax.plot(xs, norm.pdf(xs, r.theoretical_mean, r.theoretical_std_mean), 'r-', lw=2,
        label=r'Normal($\mu$, $\sigma/\sqrt{n}$)')
ax.set(title='Exponential source, n=30 — sample means look Gaussian', xlabel='value', ylabel='density')
ax.legend(); plt.show()

## 2. Convergence speed depends on skew

For heavily skewed sources (Exponential, Chi²) small n still shows visible skew. Sweep n and watch the histogram symmetrise.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, n in zip(axes, [2, 10, 30, 200], strict=True):
    r = simulate(source='exponential', n=n, num_samples=8000, seed=1)
    ax.hist(r.sample_means, bins=50, density=True, alpha=0.7)
    xs = np.linspace(r.sample_means.min(), r.sample_means.max(), 200)
    ax.plot(xs, norm.pdf(xs, r.theoretical_mean, r.theoretical_std_mean), 'r-', lw=2)
    ax.set_title(f'n = {n}')
fig.suptitle('Exponential source: skewness dies as n grows'); plt.tight_layout(); plt.show()

## 3. The √n rule

Standard error of the mean equals $\sigma / \sqrt{n}$. On a log-log plot that is a straight line with slope $-\tfrac{1}{2}$.

In [ ]:
ns = (1, 4, 16, 64, 256, 1024, 4096)
scan = sqrt_n_scan(source='uniform', ns=ns, num_samples=3000, seed=0)
sigma = np.sqrt(1/12)  # theoretical σ of Uniform(0,1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(list(scan.keys()), list(scan.values()), 'o-', label='empirical std of sample mean')
ax.loglog(ns, [sigma / np.sqrt(n) for n in ns], 'r--', label=r'theory  $\sigma/\sqrt{n}$')
ax.set(xlabel='n (log)', ylabel='std of sample mean (log)',
       title='Uniform(0,1) source: √n rule')
ax.legend(); ax.grid(True, which='both', alpha=0.3); plt.show()

## 4. What-ifs to try yourself

1. Set `source='bernoulli'`, `n=2` — the sample means take only 3 values {0, 0.5, 1}. The 'CLT' with n=2 is a very poor approximation.
2. Compare `n=25` vs `n=100` vs `n=400` for the same source — the std should shrink by ≈ 2× each step.
3. Add a Cauchy source (`rng.standard_cauchy`) and repeat — you'll see the CLT **fails**: sample means don't tighten because the variance is infinite.